[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/08-three-families.ipynb)

# Module 8: Three Families: Encoder, Encoder-Decoder, Decoder

Companion notebook to [Module 8](https://transformers-and-math.drmhse.com/modules/08-three-families) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — Train a decoder-only model on Shakespeare (section 8.4)
The causal mask lets one forward pass train every position at once. We train a tiny character-level model for a few hundred steps.

In [ ]:
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
try:
    text = urllib.request.urlopen(URL, timeout=10).read().decode()[:200_000]
except Exception:
    text = ('First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\n' * 400)
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f'{len(text):,} characters, vocabulary of {len(chars)}')

def batch(src, B=32, T=64):
    ix = torch.randint(len(src) - T - 1, (B,))
    x = torch.stack([src[i:i + T] for i in ix])
    return x, torch.stack([src[i + 1:i + T + 1] for i in ix])   # targets are the inputs shifted by one

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h, pre_ln):
        super().__init__()
        self.pre_ln = pre_ln
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        att = lambda z: self.attn(z, z, z, attn_mask=mask, need_weights=False)[0]
        if self.pre_ln:
            x = x + att(self.ln1(x))
            return x + self.ffn(self.ln2(x))
        x = self.ln1(x + att(x))
        return self.ln2(x + self.ffn(x))

class GPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, T=64, pre_ln=True):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(T, d)
        self.blocks = nn.Sequential(*[Block(d, h, pre_ln) for _ in range(L)])
        self.ln_f = nn.LayerNorm(d) if pre_ln else nn.Identity()
        self.head = nn.Linear(d, V)
    def forward(self, ids):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1]))
        return self.head(self.ln_f(self.blocks(x)))

model = GPT(len(chars))
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(301):
    x, y = batch(train_data)
    loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        with torch.no_grad():
            xv, yv = batch(val_data)
            val = F.cross_entropy(model(xv).flatten(0, 1), yv.flatten())
        print(f'step {step:3}  train {loss.item():.3f}  val {val.item():.3f}  ({val.item() / math.log(2):.2f} bits/char)')

In [ ]:
@torch.no_grad()
def generate(model, prompt, n=200, temperature=0.8):
    ids = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n):
        logits = model(ids[:, -64:])[:, -1] / temperature
        ids = torch.cat([ids, torch.multinomial(torch.softmax(logits, -1), 1)], 1)
    return ''.join(chars[i] for i in ids[0])

print(generate(model, 'ROMEO:'))

## Part 2 — Post-LN and Pre-LN at depth (section 8.5)
The gradient reaching the embeddings at initialization, as the stack gets deeper. At this size the two wirings are close; the lab in section 8.5 measured how the gap grows with depth. Pre-LN's practical advantage shows up during training: try both at 24 layers with a high learning rate and no warmup.

In [ ]:
def embedding_grad(L, pre_ln):
    torch.manual_seed(0)
    m = GPT(len(chars), L=L, pre_ln=pre_ln)
    x, y = batch(train_data, B=8)
    F.cross_entropy(m(x).flatten(0, 1), y.flatten()).backward()
    return m.tok.weight.grad.norm().item()

for L in (2, 8, 24):
    print(f'L={L:2}  Post-LN {embedding_grad(L, False):.3e}   Pre-LN {embedding_grad(L, True):.3e}')

## Part 3 — The other objective: masked language modelling (section 8.2)
BERT hides about 15% of tokens and predicts them from both sides. Here is the masking recipe.

In [ ]:
ids = data[:40].clone()
chosen = torch.rand(len(ids)) < 0.15
roll = torch.rand(len(ids))
MASK = len(chars)                                   # an extra id for [MASK]
corrupted = ids.clone()
corrupted[chosen & (roll < 0.8)] = MASK
rand_pos = chosen & (roll >= 0.8) & (roll < 0.9)
corrupted[rand_pos] = torch.randint(len(chars), (int(rand_pos.sum()),))
labels = torch.where(chosen, ids, torch.tensor(-100))   # only chosen positions count in the loss
print('chosen positions:', chosen.nonzero().flatten().tolist())

## Your turn
1. Train for 1,000 steps instead of 300. How low does the validation loss go, and how does the sample change?
2. Retrain with `pre_ln=False`. Is the difference in validation loss larger than the change you get from re-running with another seed (section 4.6)?